In [12]:
import numpy as np
from pathlib import Path
from collections import Counter

from brainvision.constants import *
from brainvision.data.io import load_processed_patients
from brainvision.validation import build_splits_fabelo, verify_no_leakage

In [13]:
# ── Load campaigns ────────────────────────────────────────────────────
def load_all_campaigns() -> dict:
    return {c: load_processed_patients(PROCESSED_DIRS[c]) for c in [1, 2, 3]}

campaigns = load_all_campaigns()

  Loaded  27 patients from ../processed/first_campaign
  Loaded  24 patients from ../processed/second_campaign
  Loaded  10 patients from ../processed/third_campaign


In [14]:
# ── Helpers ───────────────────────────────────────────────────────────
def patient_ids_from(image_list: list[dict]) -> set[str]:
    """Extract unique patient IDs from a list of image dicts."""
    return set(p['id'].split('-')[0] for p in image_list)

def image_ids_from(image_list: list[dict]) -> set[str]:
    """Extract full image IDs from a list of image dicts."""
    return set(p['id'] for p in image_list)

def summarise_splits(splits: list[dict], label: str):
    """Print a compact summary of all folds in a split list."""
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    for fold in splits:
        train_pids = patient_ids_from(fold['train'])
        val_pids   = patient_ids_from(fold['val'])
        test_pids  = patient_ids_from(fold['test'])
        print(f"  Fold {fold['fold']}: "
              f"train={len(fold['train']):2d} images ({len(train_pids):2d} pts)  "
              f"val={len(fold['val']):2d} images ({len(val_pids):2d} pts)  "
              f"test={len(fold['test']):2d} images ({len(test_pids):2d} pts)")
    print(f"  Test patients: {sorted(patient_ids_from(splits[0]['test']))}")

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEST 1 — Determinism: same seed produces identical splits every time
# ══════════════════════════════════════════════════════════════════════════════
print("TEST 1 — Determinism")
print("="*60)
print("Building splits A and B from the same seed...\n")

splits_A = build_splits_fabelo(campaigns, n_folds=5, seed=SPLIT_SEED)
splits_B = build_splits_fabelo(campaigns, n_folds=5, seed=SPLIT_SEED)

determinism_passed = True

for fold_A, fold_B in zip(splits_A, splits_B):
    train_A = image_ids_from(fold_A['train'])
    train_B = image_ids_from(fold_B['train'])
    val_A   = image_ids_from(fold_A['val'])
    val_B   = image_ids_from(fold_B['val'])
    test_A  = image_ids_from(fold_A['test'])
    test_B  = image_ids_from(fold_B['test'])

    fold_ok = (train_A == train_B and val_A == val_B and test_A == test_B)
    status  = "✅" if fold_ok else "❌ MISMATCH"
    print(f"  Fold {fold_A['fold']}: {status}")

    if not fold_ok:
        determinism_passed = False
        if train_A != train_B:
            print(f"    Train diff: {train_A.symmetric_difference(train_B)}")
        if val_A != val_B:
            print(f"    Val diff  : {val_A.symmetric_difference(val_B)}")
        if test_A != test_B:
            print(f"    Test diff : {test_A.symmetric_difference(test_B)}")

print(f"\n  Result: {'✅ PASSED' if determinism_passed else '❌ FAILED'}")

TEST 1 — Determinism
Building splits A and B from the same seed...

Total patients        : 34
Test set (fixed)      : 7 patients  (21% of X)  — shared across all folds
Pool for K-Fold CV    : 27 patients  (79% of X)
K-Fold                : 5 folds  (every pool patient appears in val exactly once)
Per fold → train      : ~21 patients  (64% of X)
Per fold → val        : ~5 patients  (15% of X)
Test patients         : ['012', '019', '022', '037', '038', '042', '053']

  Fold 1: train=34 images (21 pts)  val=12 images ( 6 pts)  test=15 images (7 pts, fixed)
  Fold 2: train=35 images (21 pts)  val=11 images ( 6 pts)  test=15 images (7 pts, fixed)
  Fold 3: train=39 images (22 pts)  val= 7 images ( 5 pts)  test=15 images (7 pts, fixed)
  Fold 4: train=39 images (22 pts)  val= 7 images ( 5 pts)  test=15 images (7 pts, fixed)
  Fold 5: train=37 images (22 pts)  val= 9 images ( 5 pts)  test=15 images (7 pts, fixed)
Total patients        : 34
Test set (fixed)      : 7 patients  (21% of X)  — sh

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEST 2 — Fixed test set: test patients are identical across all folds
# ══════════════════════════════════════════════════════════════════════════════
print("\n\nTEST 2 — Fixed test set across folds")
print("="*60)

test_sets     = [image_ids_from(fold['test']) for fold in splits_A]
fixed_test_ok = all(s == test_sets[0] for s in test_sets)

print(f"  Test patients (fold 1): {sorted(patient_ids_from(splits_A[0]['test']))}")
print()
for i, fold in enumerate(splits_A):
    matches = "✅ matches fold 1" if test_sets[i] == test_sets[0] else "❌ DIFFERS"
    print(f"  Fold {fold['fold']} test: {matches}")

print(f"\n  Result: {'✅ PASSED' if fixed_test_ok else '❌ FAILED'}")



TEST 2 — Fixed test set across folds
  Test patients (fold 1): ['012', '019', '022', '037', '038', '042', '053']

  Fold 1 test: ✅ matches fold 1
  Fold 2 test: ✅ matches fold 1
  Fold 3 test: ✅ matches fold 1
  Fold 4 test: ✅ matches fold 1
  Fold 5 test: ✅ matches fold 1

  Result: ✅ PASSED


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEST 3 — No leakage: no patient appears in more than one split per fold
# ══════════════════════════════════════════════════════════════════════════════
print("\n\nTEST 3 — No patient-level leakage per fold")
print("="*60)

leakage_passed = True
for fold in splits_A:
    try:
        verify_no_leakage(fold)
        print(f"  Fold {fold['fold']}: ✅ no leakage")
    except AssertionError as e:
        print(f"  Fold {fold['fold']}: ❌ LEAKAGE DETECTED — {e}")
        leakage_passed = False

print(f"\n  Result: {'✅ PASSED' if leakage_passed else '❌ FAILED'}")



TEST 3 — No patient-level leakage per fold
✅ No patient-level leakage detected
  Fold 1: ✅ no leakage
✅ No patient-level leakage detected
  Fold 2: ✅ no leakage
✅ No patient-level leakage detected
  Fold 3: ✅ no leakage
✅ No patient-level leakage detected
  Fold 4: ✅ no leakage
✅ No patient-level leakage detected
  Fold 5: ✅ no leakage

  Result: ✅ PASSED


In [23]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEST 4 — 60/20/20 proportions: verify split ratios match X
# ══════════════════════════════════════════════════════════════════════════════
proportions_passed = True
tolerance          = 0.08   # allow ±8% deviation from target

print(f"TEST 4 — Approximate 60/20/20 proportions (tolerance ±{tolerance*100:.0f}%)")
print(f"         Note: exact 60/20/20 not achievable with {X} patients and 5 folds")

all_patient_ids = set(
    p['id'].split('-')[0]
    for patients in campaigns.values()
    for p in patients
)
X = len(all_patient_ids)
print(f"  Total unique patients (X): {X}")

for fold in splits_A:
    train_pids = patient_ids_from(fold['train'])
    val_pids   = patient_ids_from(fold['val'])
    test_pids  = patient_ids_from(fold['test'])

    train_pct = len(train_pids) / X
    val_pct   = len(val_pids)   / X
    test_pct  = len(test_pids)  / X

    train_ok = abs(train_pct - 0.60) <= tolerance
    val_ok   = abs(val_pct   - 0.20) <= tolerance
    test_ok  = abs(test_pct  - 0.20) <= tolerance

    fold_ok  = train_ok and val_ok and test_ok
    status   = "✅" if fold_ok else "❌"

    print(f"  Fold {fold['fold']}: {status}  "
          f"train={train_pct*100:.1f}% ({len(train_pids)} pts)  "
          f"val={val_pct*100:.1f}% ({len(val_pids)} pts)  "
          f"test={test_pct*100:.1f}% ({len(test_pids)} pts)")

    if not fold_ok:
        proportions_passed = False
        if not train_ok:
            print(f"    ❌ Train {train_pct*100:.1f}% deviates from 60% "
                  f"by more than {tolerance*100:.0f}%")
        if not val_ok:
            print(f"    ❌ Val {val_pct*100:.1f}% deviates from 20% "
                  f"by more than {tolerance*100:.0f}%")
        if not test_ok:
            print(f"    ❌ Test {test_pct*100:.1f}% deviates from 20% "
                  f"by more than {tolerance*100:.0f}%")

print(f"\n  Result: {'✅ PASSED' if proportions_passed else '❌ FAILED'}")

TEST 4 — Approximate 60/20/20 proportions (tolerance ±8%)
         Note: exact 60/20/20 not achievable with 34 patients and 5 folds
  Total unique patients (X): 34
  Fold 1: ✅  train=61.8% (21 pts)  val=17.6% (6 pts)  test=20.6% (7 pts)
  Fold 2: ✅  train=61.8% (21 pts)  val=17.6% (6 pts)  test=20.6% (7 pts)
  Fold 3: ✅  train=64.7% (22 pts)  val=14.7% (5 pts)  test=20.6% (7 pts)
  Fold 4: ✅  train=64.7% (22 pts)  val=14.7% (5 pts)  test=20.6% (7 pts)
  Fold 5: ✅  train=64.7% (22 pts)  val=14.7% (5 pts)  test=20.6% (7 pts)

  Result: ✅ PASSED


In [19]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEST 5 — Fold diversity: each fold has a different val set
# ══════════════════════════════════════════════════════════════════════════════
print("\n\nTEST 5 — Fold diversity (each fold has a different val set)")
print("="*60)

val_sets       = [patient_ids_from(fold['val']) for fold in splits_A]
diversity_ok   = True

for i in range(len(val_sets)):
    for j in range(i+1, len(val_sets)):
        overlap = val_sets[i] & val_sets[j]
        pct     = len(overlap) / len(val_sets[i]) * 100
        status  = "✅" if len(overlap) < len(val_sets[i]) else "❌ IDENTICAL"
        print(f"  Fold {i+1} vs Fold {j+1}: "
              f"{len(overlap)} shared val patients ({pct:.0f}% overlap)  {status}")
        if len(overlap) == len(val_sets[i]):
            diversity_ok = False

print(f"\n  Result: {'✅ PASSED' if diversity_ok else '❌ FAILED'}")



TEST 5 — Fold diversity (each fold has a different val set)
  Fold 1 vs Fold 2: 0 shared val patients (0% overlap)  ✅
  Fold 1 vs Fold 3: 0 shared val patients (0% overlap)  ✅
  Fold 1 vs Fold 4: 0 shared val patients (0% overlap)  ✅
  Fold 1 vs Fold 5: 0 shared val patients (0% overlap)  ✅
  Fold 2 vs Fold 3: 0 shared val patients (0% overlap)  ✅
  Fold 2 vs Fold 4: 0 shared val patients (0% overlap)  ✅
  Fold 2 vs Fold 5: 0 shared val patients (0% overlap)  ✅
  Fold 3 vs Fold 4: 0 shared val patients (0% overlap)  ✅
  Fold 3 vs Fold 5: 0 shared val patients (0% overlap)  ✅
  Fold 4 vs Fold 5: 0 shared val patients (0% overlap)  ✅

  Result: ✅ PASSED


In [20]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEST 6 — Different seed: produces different splits
# ══════════════════════════════════════════════════════════════════════════════
print("\n\nTEST 6 — Different seed produces different splits")
print("="*60)

splits_C     = build_splits_fabelo(campaigns, n_folds=5, seed=SPLIT_SEED + 99)
different_ok = False

for fold_A, fold_C in zip(splits_A, splits_C):
    if image_ids_from(fold_A['train']) != image_ids_from(fold_C['train']):
        different_ok = True
        break

test_different = (image_ids_from(splits_A[0]['test']) !=
                  image_ids_from(splits_C[0]['test']))

print(f"  Different train sets   : {'✅ Yes' if different_ok   else '❌ Identical — seed has no effect'}")
print(f"  Different test sets    : {'✅ Yes' if test_different  else '⚠️  Same test set (possible with small dataset)'}")
print(f"\n  Result: {'✅ PASSED' if different_ok else '❌ FAILED'}")



TEST 6 — Different seed produces different splits
Total patients        : 34
Test set (fixed)      : 7 patients  (21% of X)  — shared across all folds
Pool for K-Fold CV    : 27 patients  (79% of X)
K-Fold                : 5 folds  (every pool patient appears in val exactly once)
Per fold → train      : ~21 patients  (64% of X)
Per fold → val        : ~5 patients  (15% of X)
Test patients         : ['008', '010', '013', '019', '051', '053', '057']

  Fold 1: train=41 images (21 pts)  val=12 images ( 6 pts)  test= 8 images (7 pts, fixed)
  Fold 2: train=42 images (21 pts)  val=11 images ( 6 pts)  test= 8 images (7 pts, fixed)
  Fold 3: train=42 images (22 pts)  val=11 images ( 5 pts)  test= 8 images (7 pts, fixed)
  Fold 4: train=41 images (22 pts)  val=12 images ( 5 pts)  test= 8 images (7 pts, fixed)
  Fold 5: train=46 images (22 pts)  val= 7 images ( 5 pts)  test= 8 images (7 pts, fixed)
  Different train sets   : ✅ Yes
  Different test sets    : ✅ Yes

  Result: ✅ PASSED


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
#  TEST 7 — Coverage: every patient in pool appears in val at least once
# ══════════════════════════════════════════════════════════════════════════════
print("\n\nTEST 7 — Pool coverage (every pool patient appears in val at least once)")
print("="*60)

test_pids = patient_ids_from(splits_A[0]['test'])
pool_pids = all_patient_ids - test_pids

val_counts = Counter()
for fold in splits_A:
    for pid in patient_ids_from(fold['val']):
        val_counts[pid] += 1

never_val = pool_pids - set(val_counts.keys())
coverage  = (len(never_val) == 0)

print(f"  Pool patients          : {len(pool_pids)}")
print(f"  Patients in val ≥1x   : {len(val_counts)}")
print(f"  Never appeared in val  : {len(never_val)}")
if never_val:
    print(f"  Missing patients       : {sorted(never_val)}")

print(f"\n  Val appearance counts:")
for pid, count in sorted(val_counts.items()):
    bar = '█' * count
    print(f"    {pid}: {bar} ({count}x)")

print(f"\n  Result: {'✅ PASSED' if coverage else '⚠️  Some pool patients never appear in val'}")




TEST 7 — Pool coverage (every pool patient appears in val at least once)
  Pool patients          : 27
  Patients in val ≥1x   : 27
  Never appeared in val  : 0

  Val appearance counts:
    004: █ (1x)
    005: █ (1x)
    007: █ (1x)
    008: █ (1x)
    010: █ (1x)
    013: █ (1x)
    014: █ (1x)
    015: █ (1x)
    016: █ (1x)
    017: █ (1x)
    018: █ (1x)
    020: █ (1x)
    021: █ (1x)
    034: █ (1x)
    035: █ (1x)
    036: █ (1x)
    039: █ (1x)
    040: █ (1x)
    041: █ (1x)
    043: █ (1x)
    050: █ (1x)
    051: █ (1x)
    054: █ (1x)
    055: █ (1x)
    056: █ (1x)
    057: █ (1x)
    058: █ (1x)

  Result: ✅ PASSED


In [22]:
# ── Final summary ────────────────────────────────────────────────────
print("\n\n" + "="*60)
print("  FINAL TEST SUMMARY")
print("="*60)

tests = {
    "Determinism (same seed → same splits)"  : determinism_passed,
    "Fixed test set across all folds"        : fixed_test_ok,
    "No patient-level leakage"              : leakage_passed,
    "60/20/20 proportions (±5%)"            : proportions_passed,
    "Fold diversity (different val sets)"    : diversity_ok,
    "Different seed → different splits"      : different_ok,
    "Pool coverage (all patients in val)"    : coverage,
}

all_passed = all(tests.values())

for test_name, passed in tests.items():
    status = "✅ PASSED" if passed else "❌ FAILED"
    print(f"  {status}  {test_name}")

print(f"\n{'='*60}")
print(f"  {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")
print(f"{'='*60}")



  FINAL TEST SUMMARY
  ✅ PASSED  Determinism (same seed → same splits)
  ✅ PASSED  Fixed test set across all folds
  ✅ PASSED  No patient-level leakage
  ✅ PASSED  60/20/20 proportions (±5%)
  ✅ PASSED  Fold diversity (different val sets)
  ✅ PASSED  Different seed → different splits
  ✅ PASSED  Pool coverage (all patients in val)

  ✅ ALL TESTS PASSED
